In [2]:
#Amy Independent Research, Fall 2024
import networkx as nx
import osmnx as ox
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import ScrollZoomToggler
import matplotlib.pyplot as plt
import shapely
from glob import glob

import osmnx.settings as settings
import osmnx.features as features

ox.__version__

settings.cache_folder = "/tmp/cache"

For each model (Model 1 to Model 8), these are the regressors.
Dependent Variable: Average Daily Trip Counts (avgdtc)
1) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Intercept
2) Independent Variables:  Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Electric Bike Proportion (elecbikep), Intercept
3) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Electric Bike Proportion (elecbikep), Electric Bike Proportion * Temperature (elecbikep_avgt), Intercept
4) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Membership Proportion (memberp), Intercept
5) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Membership Proportion (memberp), Membership Proportion * Bike Lane Length (memberp_bl), Intercept
6) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Membership Proportion (memberp), Electric Bike Proportion (elecbikep), Electric Bike Proportion * Temperature (elecbikep_avgt), Membership Proportion * Bike Lane Length (memberp_bl), Membership Proportion * Electric Bike Proportion (memberp_elecbikep), Intercept
7) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Active Stations (activesta), Gravity Attractiveness, Intercept

avgdtc: average daily trip counts; avgpp: average precipitation (inch); avgsd: average snow depth (in); avgt: average temperature (F); avgws: average wind speed (mph); bl: total bike lane length (mile); activesta – average number of daily active stations; elecbikep - electric bike proportion; elecbikep_avgt - interaction term between electric bike proportion and average temperature (F); memberp - Membership Proportion; memberp_bl - interaction term between membership proportion and bike lane length (mile); memberp_elecbikep - interaction term between membership proportion and electric bike proportion

# AR² refers to the autoregressive process where the current value of the dependent variable (Average Daily Trip Counts) depends on its lagged values up to the second order (2 previous time steps). It captures the effect of temporal autocorrelation in the dependent variable. For example, high trip counts on one day might be followed by high trip counts on the next day (due to momentum or recurring patterns).
# ARCH³ refers to the third-order autoregressive conditional heteroskedasticity component of the model. ARCH³ models the conditional variance of the residuals (𝜎t squared) as a function of the squared residuals from up to 3 previous time steps. _cons (Intercept) represents the constant term in the model. It accounts for the baseline level of average daily trip counts when all other variables (independent variables and lagged terms) are zero.
# The log-likelihood measures how well the model fits the data. Higher values indicate a better fit.
# The Likelihood Ratio Test (LRT) compares the goodness-of-fit between two nested models (e.g., one model is a simplified version of the other). Tests whether adding parameters (e.g., additional lag terms or ARCH terms) significantly improves the model. Adding more lags or ARCH terms should increase the log-likelihood if they improve model fit. A significant p-value for the LRT indicates that the more complex model (with additional parameters) provides a significantly better fit.

Model 1: Observation on average over seven consecutive days in New York City. (ARCH)

Model 2: Observation on average over seven consecutive days in Non-Manhattan.

Model 3: Observation on average over seven consecutive days in Manhattan.

Model 4: Only average weekdays are included in each observation in Manhattan. 

Model 5: Only average weekends are included in each observation in Manhattan.

Model 6: Observation on average over seven consecutive days in Brooklyn.

Model 7: Only average weekdays are included in each observation in Brooklyn. 

Model 8: Only average weekends are included in each observation in Brooklyn.

In [ ]:
# Define the folder path where the files are located
folder_path = "/Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike"

# Create an empty list to store DataFrames
weekly_dataframes = []

# Loop through each year and month from 2021 to 2024 (up to and including November)
for year in range(2021, 2024):
    for month in range(1, 13):
        # Skip months beyond September in 2024
        if year == 2024 and month > 11:
            continue

        # Format the month as two digits
        formatted_month = f"{month:02d}"

        # Construct the file pattern to match all parts (e.g., _1, _2, etc.)
        file_pattern = os.path.join(folder_path, f"{year}{formatted_month}-citibike-tripdata_*.csv")
        
        # Use glob to find all files matching the pattern
        file_list = glob(file_pattern)

        # Debug: Print the files found
        print(f"Looking for files in: {file_pattern}")
        print(f"Files found: {file_list}")

        # Process each file found for this month
        for file_name in file_list:
            print(f"Processing file: {file_name}")

            # Load the CSV file into a DataFrame
            df = pd.read_csv(file_name, parse_dates=['started_at'])

            # Debug: Check the number of rows loaded
            print(f"Number of rows read from {file_name}: {len(df)}")

            # Skip empty DataFrames
            if df.empty:
                print(f"File {file_name} is empty. Skipping.")
                continue

            # Ensure 'started_at' is a datetime object
            df['start_date'] = pd.to_datetime(df['started_at'])

            # Set 'start_date' as the index for resampling
            df.set_index('start_date', inplace=True)

            # Resample to weekly, counting the number of rides per week
            weekly_rides = df.resample('W')['ride_id'].count().reset_index()
            weekly_rides.rename(columns={'ride_id': 'total_rides'}, inplace=True)

            # Add the year and week number columns using the isocalendar function
            weekly_rides['year'] = weekly_rides['start_date'].dt.year
            weekly_rides['week_number'] = weekly_rides['start_date'].dt.isocalendar().week

            # Add the weekly data to the list
            weekly_dataframes.append(weekly_rides)

# Combine all the weekly DataFrames into a single DataFrame
if weekly_dataframes:
    combined_weekly_data = pd.concat(weekly_dataframes, ignore_index=True)
    # Save the combined data to a CSV file
    output_file = os.path.join(folder_path, "citibike_weekly_ridership.csv")
    combined_weekly_data.to_csv(output_file, index=False)
    print(f"Weekly aggregated data saved to {output_file}")
else:
    print("No weekly data was generated. Please check your file paths and data.")